# 🚀 Huấn luyện & Tự động nộp Submission 16 cho DS102 trên Google Colab
Notebook này được tối ưu sẵn để chạy trên **Google Colab (GPU T4 miễn phí / 16GB VRAM)**:
1. **Tốc độ vượt trội:** GPU 16GB VRAM không bị nghẽn bộ nhớ, chạy cả 5-fold Context-XLM chỉ mất ~20–25 phút (thay vì bị Windows Modern Standby hạ xung trên máy tính cá nhân).
2. **Tự động hoá toàn diện:** Huấn luyện xong sẽ tự động chạy tổ hợp Stacking 80 cây quyết định, tạo file `submission_16.csv` và **tự động nộp thẳng lên Kaggle bằng CLI** rồi in điểm Public Leaderboard.

In [1]:
# 1. Kiểm tra GPU của Colab (khuyên dùng Runtime > Change runtime type > T4 GPU)
!nvidia-smi

Fri Nov  6 18:56:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             49W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# 2. Clone repository chứa mã nguồn và các ma trận đặc trưng OOF đã tính sẵn
import os
if not os.path.exists("KaggleComp"):
    !git clone https://github.com/24520559-Nh0c3c/KaggleComp.git
%cd KaggleComp
!git pull origin main

Cloning into 'KaggleComp'...
remote: Enumerating objects: 130, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 130 (delta 23), reused 123 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (130/130), 29.25 MiB | 30.07 MiB/s, done.
Resolving deltas: 100% (23/23), done.
/content/KaggleComp
From https://github.com/24520559-Nh0c3c/KaggleComp
 * branch            main       -> FETCH_HEAD
Already up to date.


In [3]:
# 3. Cài đặt các thư viện cần thiết
!pip install -q transformers sentencepiece lightgbm xgboost catboost kaggle

In [4]:
# 4. Cấu hình xác thực Kaggle API để tự động nộp bài
import os
from google.colab import files

if not os.path.exists("/root/.kaggle/kaggle.json"):
    print("👉 Hãy tải file kaggle.json của bạn lên (lấy tại kaggle.com/settings/api):")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith(".json"):
            !mkdir -p /root/.kaggle
            !cp "$fn" /root/.kaggle/kaggle.json
            !chmod 600 /root/.kaggle/kaggle.json
            print("✅ Cấu hình Kaggle API thành công!")
            break
else:
    print("✅ Kaggle API đã sẵn sàng!")

👉 Hãy tải file kaggle.json của bạn lên (lấy tại kaggle.com/settings/api):


Saving kaggle.json to kaggle.json
✅ Cấu hình Kaggle API thành công!


In [5]:
# 5. Huấn luyện 5-Fold Context-Aware XLM-RoBERTa (Variant B: Product-First + 50-word window, seed 2024)
# Trên GPU T4 16GB VRAM, tiến trình chỉ mất khoảng ~20-25 phút cho toàn bộ 5 Fold!
!python train_context_xlm.py --variant b --seed 2024 --tag xlm_ctx2

Using device: cuda | Smoke: False
Building context-aware NFKC inputs for Train, Dev, and Test...
Example input: Nước Xả Làm Mềm Vải Comfort Chăm Sóc Dịu Nhẹ Cho Da Nhạy Cảm 3.2L | Giặt giũ & Chăm sóc quần áo > Nước xả vải | 5 sao </s> sản phẩm đúng quảng cáo. giao hàng hơi chậm nhg vẫn cho 5 sao </s> quần áo trong 10 lít nước Lưu ý: Bảo quản sản phẩm ở nơi thoáng mát và tránh ánh nắng trực tiếpGiá sản phẩm tr
config.json: 100% 615/615 [00:00<00:00, 2.56MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 106kB/s]
sentencepiece.bpe.model: 100% 5.07M/5.07M [00:00<00:00, 9.76MB/s]
tokenizer.json: 100% 9.10M/9.10M [00:00<00:00, 27.5MB/s]
Tokenized 15896 train+dev texts: mean_len=151.6, max_len=256

model.safetensors: downloading bytes:  14% 151M/1.12G [00:01<00:03, 259MB/s, 7.43MB/s  ] 
model.safetensors: reconstructing file:   6% 67.0M/1.12G [00:01<00:17, 58.8MB/s]
model.safetensors: downloading bytes:  34% 379M/1.12G [00:01<00:01, 476MB/s, 29.2MB/s  ]
model.safetensors: reconstructi

In [6]:
# 6. Chạy Stacking 80 cây quyết định kết hợp với neo xác suất Submission 15 (0.80393)
# Tự động tạo submission_16.csv và gọi Kaggle CLI nộp bài trực tiếp lên bảng xếp hạng!
!python build_submission_16.py

1. Loading datasets, 6 Level-1 probability streams, and KNN/SVD features...
   Using averaged Context-XLM (run A + run B)
2. Building 233-feature Dual-Context + KNN/SVD + Copy-Paste matrices...
   Total feature count: 233
3. Training Stage A (5-Fold CatBoost-d6/d5 + Reg-LGBM + Reg-XGB on Train)...
   [Stage A CAT6] Raw OOF F1: 0.80444 | DEV OOF F1: 0.73780
   [Stage A CAT5] Raw OOF F1: 0.79970 | DEV OOF F1: 0.74614
   [Stage A LGB ] Raw OOF F1: 0.80351 | DEV OOF F1: 0.73229
   [Stage A XGB ] Raw OOF F1: 0.80307 | DEV OOF F1: 0.72881
4. Training Stage B (10-Fold 60-Model Bagged Ensemble on Full Train+Dev = 15,896 rows)...
   [Stage B CAT6] Raw Train-Slice F1: 0.80612 | Full 15,896 OOF F1: 0.80234
   [Stage B CAT5] Raw Train-Slice F1: 0.80815 | Full 15,896 OOF F1: 0.80352
   [Stage B LGB ] Raw Train-Slice F1: 0.80950 | Full 15,896 OOF F1: 0.80454
   [Stage B XGB ] Raw Train-Slice F1: 0.81164 | Full 15,896 OOF F1: 0.80605

Nested-CV Averaged Calibration Weights:  [1.0, 1.078, 0.998, 0.988

In [7]:
# 7. Tải file submission_16.csv về máy tính của bạn để lưu trữ
from google.colab import files
if os.path.exists("submission_16.csv"):
    files.download("submission_16.csv")
    print("✅ Đã tải file nộp submission_16.csv về máy!")
else:
    print("Chưa tìm thấy submission_16.csv, hãy kiểm tra lại bước 6.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Đã tải file nộp submission_16.csv về máy!
